# Rasmchi AI — FLUX Colab (yuqori sifat)\nBu notebookda `black-forest-labs/FLUX.1-schnell` 4-bit kvantizatsiya bilan ishga tushiriladi va Gradio public URL orqali saytga ulash mumkin.\n\n**Tartib:**\n1. GPU'ni yoqib, quyidagi cell'ni ishlating.\n2. `Runtime → Restart session` bosing.\n3. Keyin `Sinash` bo'limidagi cell'larni ishga tushiring.\n4. Hosil bo'lgan `gradio.live` URL'sini saytdagi "Mening Colab'im" qatoriga kiring.

## 1. O'rnatish

In [ ]:
!pip install -q -U bitsandbytes diffusers transformers accelerate gradio deep-translator

**To'xtating va `Runtime → Restart session` bosing.**\nAks holda eski `bitsandbytes` yuklanib qolib, xato beradi.

## 2. Sinash

In [ ]:
import random, torch, gradio as gr\nfrom diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DBnb\nfrom transformers import T5EncoderModel, BitsAndBytesConfig as TBnb\nfrom huggingface_hub import login\nfrom deep_translator import GoogleTranslator\n\nHF_TOKEN = ""\ntry:\n    from google.colab import userdata\n    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN\nexcept Exception:\n    pass\n\nif not HF_TOKEN:\n    raise ValueError("HF_TOKEN kerak (Colab Secrets → HF_TOKEN)")\n\nlogin(token=HF_TOKEN)\n\nrepo, dt = "black-forest-labs/FLUX.1-schnell", torch.bfloat16\ntf = FluxTransformer2DModel.from_pretrained(\n    repo, subfolder="transformer",\n    quantization_config=DBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dt),\n    torch_dtype=dt\n)\nte = T5EncoderModel.from_pretrained(\n    repo, subfolder="text_encoder_2",\n    quantization_config=TBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dt),\n    torch_dtype=dt\n)\npipe = FluxPipeline.from_pretrained(repo, transformer=tf, text_encoder_2=te, torch_dtype=dt)\npipe.enable_model_cpu_offload()\n\ndef tr(t):\n    if not t or not t.strip():\n        return ""\n    try:\n        return GoogleTranslator(source="auto", target="en").translate(t)\n    except Exception:\n        return t\n\ndef make_logo(text, style_uz, steps):\n    style = tr(style_uz) or "modern, minimalist, clean, professional"\n    if text and text.strip():\n        p = f'professional logo design with the text \{text.strip()}\, {style}, vector style, centered, clean background, high quality, sharp'\n    else:\n        p = f'professional logo design, {style}, vector style, centered, clean background'\n    seed = random.randint(0, 2**31-1)\n    g = torch.Generator("cpu").manual_seed(seed)\n    img = pipe(p, num_inference_steps=int(steps), guidance_scale=0.0, width=1024, height=1024, generator=g).images[0]\n    return img, f"\{p} · seed: \{seed}"\n\nwith gr.Blocks(title="Rasmchi AI — FLUX Logo", theme=gr.themes.Soft(primary_hue="purple")) as demo:\n    gr.Markdown("# Rasmchi AI — FLUX (yuqori sifat)\")\n    with gr.Row():\n        with gr.Column():\n            text = gr.Textbox(label="Logotipda yoziladigan matn", value="Akobir")\n            style_uz = gr.Textbox(label="Uslub (o\'zbekcha)", lines=3, placeholder="qora fon, kumushrang metall harflar, dumaloq emblema, hashamatli")\n            steps = gr.Slider(2, 8, value=4, step=1, label="Qadamlar")\n            btn = gr.Button("Logo yaratish", variant="primary")\n        with gr.Column():\n            out_img = gr.Image(label="Natija", type="pil")\n            out_txt = gr.Markdown()\n    btn.click(make_logo, [text, style_uz, steps], [out_img, out_txt])\n\ndemo.queue().launch(share=True)\n

## 3. Saytga ulash\nYuqoridagi cell'ni ishlatib, `gradio.live` yoki `ngrok` tarzidagi public URL chiqqach, uni saytdagi "Mening Colab'im" qatoriga kiritib yuklang.\n\nUshbu URL hech kimga tarqatmang, chunki u sizning Colab GPU vaqutingizdan foydalanadi.